# Hex6 Colab Training

This notebook can run bootstrap training, longer cycle training, a search-matrix eval,
a round-robin tournament, or a priority-scored queue loop in Google Colab.
Use `configs/fast.toml` only when you want the quickest possible smoke test.

If `HEX6_GITHUB_TOKEN` is set in Colab secrets, the run will publish live status JSON
to the `colab-status` branch so the local watcher can detect completion.

In [ ]:
REPO_MODE = "git"  # set to "drive" to use a repo folder in Google Drive
REPO_URL = "https://github.com/Stroudmj00/hex6-bot.git"
DRIVE_REPO_PATH = "/content/drive/MyDrive/Hexagonal tic tac toe"
WORKDIR = "/content/hex6-bot"
RUN_MODE = "priority_loop"  # "bootstrap", "cycle", "search_matrix", "tournament", or "priority_loop"
REQUIRE_COLAB = True
REQUIRE_GPU = True
EXPECTED_BRANCH = "main"
ENFORCE_LATEST_MAIN = True
CONFIG_PATH = "configs/colab_hour.toml"
WATCH_CONFIG_PATH = "configs/colab.toml"
MATRIX_PATH = "configs/experiments/search_matrix.toml"
PRIORITY_QUEUE_PATH = "configs/colab_job_queue.toml"
PRIORITY_QUEUE_STATE = "artifacts/colab_queue/state.json"
PRIORITY_QUEUE_MAX_MINUTES = None  # set an int (for example 480) if you want bounded runs
TOURNAMENT_CONFIG_PATH = "configs/fast.toml"
TOURNAMENT_GAMES_PER_MATCH = 2
TOURNAMENT_MAX_GAME_PLIES = 100
TOURNAMENT_CHECKPOINT_GLOB = "artifacts/**/bootstrap_model.pt"
TOURNAMENT_MAX_CHECKPOINTS = 3
TOURNAMENT_INCLUDE_BASELINE = True
TOURNAMENT_INCLUDE_RANDOM = True
TOURNAMENT_RANDOM_SEED = 7
DEFAULT_OUTPUT_DIRS = {
    "bootstrap": "artifacts/bootstrap_colab",
    "cycle": "artifacts/bootstrap_colab_hour",
    "search_matrix": "artifacts/search_matrix_colab",
    "tournament": "artifacts/tournament_colab",
    "priority_loop": "artifacts/colab_queue",
}
DEFAULT_ARTIFACT_TARGETS = {
    "bootstrap": "/content/drive/MyDrive/hex6_artifacts/bootstrap_colab",
    "cycle": "/content/drive/MyDrive/hex6_artifacts/bootstrap_colab_hour",
    "search_matrix": "/content/drive/MyDrive/hex6_artifacts/search_matrix_colab",
    "tournament": "/content/drive/MyDrive/hex6_artifacts/tournament_colab",
    "priority_loop": "/content/drive/MyDrive/hex6_artifacts/colab_queue",
}
OUTPUT_DIR = DEFAULT_OUTPUT_DIRS[RUN_MODE]
CYCLE_MINUTES = 60
ARTIFACT_TARGET = DEFAULT_ARTIFACT_TARGETS[RUN_MODE]


In [ ]:
import json
import os
import shutil
import subprocess
from datetime import datetime, timezone
from pathlib import Path

RUN_ID = f"colab-{datetime.now(timezone.utc):%Y%m%d-%H%M%S}"
print("RUN_ID:", RUN_ID)

try:
    from google.colab import userdata
    in_colab = True
except Exception:
    userdata = None
    in_colab = False

if REQUIRE_COLAB and not in_colab:
    raise RuntimeError("This notebook is configured for Google Colab. Open it in Colab or set REQUIRE_COLAB=False.")

print("IN_COLAB:", in_colab)

if userdata is not None:
    try:
        token = userdata.get("HEX6_GITHUB_TOKEN")
    except Exception:
        token = None
else:
    token = None

if token:
    os.environ["HEX6_GITHUB_TOKEN"] = token
    print("Loaded HEX6_GITHUB_TOKEN from Colab secrets.")
else:
    print("No HEX6_GITHUB_TOKEN secret found. GitHub status publishing will fail if enabled.")

if REPO_MODE == "drive":
    from google.colab import drive
    drive.mount('/content/drive')
    if os.path.exists(WORKDIR):
        shutil.rmtree(WORKDIR)
    shutil.copytree(DRIVE_REPO_PATH, WORKDIR)
else:
    if os.path.exists(WORKDIR):
        shutil.rmtree(WORKDIR)
    subprocess.run(["git", "clone", REPO_URL, WORKDIR], check=True)

os.chdir(WORKDIR)
print("WORKDIR:", os.getcwd())


def _run_capture(cmd: list[str]) -> tuple[int, str, str]:
    result = subprocess.run(cmd, check=False, capture_output=True, text=True)
    return result.returncode, result.stdout.strip(), result.stderr.strip()


repo_version = {
    "run_id": RUN_ID,
    "branch": "",
    "head": "",
    "head_short": "",
    "head_commit_utc": "",
    "notebook_last_modified_utc": "",
    "remote_head": "",
    "is_latest_main": None,
    "checked_at_utc": datetime.now(timezone.utc).isoformat(),
}

if Path(".git").exists():
    _, branch, _ = _run_capture(["git", "rev-parse", "--abbrev-ref", "HEAD"])
    _, head, _ = _run_capture(["git", "rev-parse", "HEAD"])
    _, head_short, _ = _run_capture(["git", "rev-parse", "--short", "HEAD"])
    _, head_commit_utc, _ = _run_capture(["git", "log", "-1", "--format=%cI"])
    _, notebook_last_modified_utc, _ = _run_capture([
        "git",
        "log",
        "-1",
        "--format=%cI",
        "--",
        "notebooks/hex6_colab_fast_bootstrap.ipynb",
    ])

    repo_version.update(
        {
            "branch": branch,
            "head": head,
            "head_short": head_short,
            "head_commit_utc": head_commit_utc,
            "notebook_last_modified_utc": notebook_last_modified_utc,
        }
    )

    remote_head = ""
    fetch_code, _, _ = _run_capture(["git", "fetch", "origin", EXPECTED_BRANCH, "--quiet"])
    if fetch_code == 0:
        _, remote_head, _ = _run_capture(["git", "rev-parse", f"origin/{EXPECTED_BRANCH}"])
        repo_version["remote_head"] = remote_head
        repo_version["is_latest_main"] = bool(head and remote_head and head == remote_head)
    else:
        print("Warning: unable to fetch origin for freshness check.")

    if ENFORCE_LATEST_MAIN and repo_version["is_latest_main"] is False:
        raise RuntimeError(
            "Repository is not at latest origin/main. Re-run with REPO_MODE='git' or sync your Drive copy before launching."
        )
else:
    print("Warning: .git metadata missing; freshness check skipped.")

print("Repo version:")
print(json.dumps(repo_version, indent=2))
print("Local watch command:")
watch_command = f".\.venv\Scripts\python -m hex6.integration.watch_status --config {WATCH_CONFIG_PATH} --run-id {RUN_ID}"
if token:
    watch_command += " --status-backend github_branch"
print(watch_command)


In [ ]:
subprocess.run(["python", "-m", "pip", "install", "-U", "pip"], check=True)
subprocess.run(["python", "-m", "pip", "install", "numpy"], check=True)
subprocess.run(["python", "-m", "pip", "install", "-e", ".[dev]"], check=True)

import torch

print("torch:", torch.__version__)
cuda_ok = torch.cuda.is_available()
print("cuda_available:", cuda_ok)
if cuda_ok:
    device_name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    total_gb = props.total_memory / (1024 ** 3)
    print("gpu:", device_name)
    print("gpu_memory_gb:", round(total_gb, 2))

if REQUIRE_GPU and not cuda_ok:
    raise RuntimeError("No CUDA GPU found. In Colab, switch Runtime -> Change runtime type -> GPU.")


In [ ]:
if RUN_MODE == "cycle":
    command = [
        "python",
        "-m",
        "hex6.train.run_cycle",
        "--config",
        CONFIG_PATH,
        "--output-root",
        OUTPUT_DIR,
        "--minutes",
        str(CYCLE_MINUTES),
        "--run-id",
        RUN_ID,
    ]
elif RUN_MODE == "bootstrap":
    command = [
        "python",
        "-m",
        "hex6.train.run_bootstrap",
        "--config",
        CONFIG_PATH,
        "--output",
        OUTPUT_DIR,
        "--run-id",
        RUN_ID,
    ]
elif RUN_MODE == "search_matrix":
    command = [
        "python",
        "-m",
        "hex6.eval.run_search_matrix",
        "--matrix",
        MATRIX_PATH,
        "--output",
        OUTPUT_DIR,
        "--run-id",
        RUN_ID,
    ]
elif RUN_MODE == "tournament":
    command = [
        "python",
        "-m",
        "hex6.eval.run_tournament",
        "--config",
        TOURNAMENT_CONFIG_PATH,
        "--output",
        OUTPUT_DIR,
        "--games-per-match",
        str(TOURNAMENT_GAMES_PER_MATCH),
        "--max-game-plies",
        str(TOURNAMENT_MAX_GAME_PLIES),
        "--checkpoint-glob",
        TOURNAMENT_CHECKPOINT_GLOB,
        "--max-checkpoints",
        str(TOURNAMENT_MAX_CHECKPOINTS),
        "--random-seed",
        str(TOURNAMENT_RANDOM_SEED),
        "--run-id",
        RUN_ID,
    ]
    if not TOURNAMENT_INCLUDE_BASELINE:
        command.append("--no-include-baseline")
    if not TOURNAMENT_INCLUDE_RANDOM:
        command.append("--no-include-random")
elif RUN_MODE == "priority_loop":
    command = [
        "python",
        "-m",
        "hex6.integration.run_priority_loop",
        "--queue",
        PRIORITY_QUEUE_PATH,
        "--state",
        PRIORITY_QUEUE_STATE,
    ]
    if PRIORITY_QUEUE_MAX_MINUTES is not None:
        command.extend(["--max-minutes", str(PRIORITY_QUEUE_MAX_MINUTES)])
else:
    raise ValueError(f"unsupported RUN_MODE: {RUN_MODE}")
if token:
    command.extend(["--status-backend", "github_branch"])
else:
    command.extend(["--status-backend", "none"])
subprocess.run(command, check=True)


In [ ]:
if RUN_MODE == "cycle":
    result_path = Path(OUTPUT_DIR) / "cycle_summary.json"
elif RUN_MODE == "bootstrap":
    result_path = Path(OUTPUT_DIR) / "metrics.json"
elif RUN_MODE == "priority_loop":
    result_path = Path(PRIORITY_QUEUE_STATE)
else:
    result_path = Path(OUTPUT_DIR) / "summary.json"
print(result_path.read_text())
print("Status file:", "https://raw.githubusercontent.com/Stroudmj00/hex6-bot/colab-status/status/latest.json")

output_path = Path(OUTPUT_DIR)
output_path.mkdir(parents=True, exist_ok=True)
version_path = output_path / "repo_version.json"
version_path.write_text(json.dumps(repo_version, indent=2), encoding="utf-8")
print("Version metadata:", version_path)

if REPO_MODE == "drive":
    target = Path(ARTIFACT_TARGET)
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        shutil.rmtree(target)
    shutil.copytree(OUTPUT_DIR, target)
    print("Artifacts copied to", target)
